# Multi-Layer Phishing Detection & Risk Assessment
## Machine Learning Model Development, Evaluation & Risk Assessment
**Contributors:** Siddhi Shetkar & Sankeerthana Thoviti

---

### Core System Workflow:
1. **Model Development:** Train 5 distinct machine learning classification algorithms:
   - Logistic Regression (Linear baseline with StandardScaling)
   - Decision Tree Classifier
   - Random Forest Classifier (Ensemble bagging)
   - Linear Support Vector Machine (with Calibrated Probability Scores)
   - Histogram-based Gradient Boosting (Ensemble boosting)
2. **Rigorous Evaluation:** Compare models on held-out test data (40,000 samples) across Accuracy, Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrices.
3. **Multi-Layer Risk Assessment:** Establish configurable risk tiers (LOW, MEDIUM, HIGH) combining model probabilities with heuristic security threat indicators.
4. **End-to-End Prediction:** Provide an explainable URL safety classification system that runs offline without accessing untrusted networks.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is accessible
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_extraction import FEATURE_COLUMNS, extract_url_features
from src.model import get_model_candidates, train_model, evaluate_model, save_model_artifacts, load_model, predict_single_url_features
from src.risk_assessment import RiskAssessmentEngine, assess_url_risk

print(f"Setup complete. Using {len(FEATURE_COLUMNS)} engineered features.")

## 1. Load Preprocessed Dataset Splits
We load the stratified 80/20 train/test split created without data leakage.

In [ ]:
splits_dir = PROJECT_ROOT / "dataset" / "processed" / "splits"

X_train = pd.read_csv(splits_dir / "X_train.csv")
y_train = pd.read_csv(splits_dir / "y_train.csv").squeeze("columns")
X_test = pd.read_csv(splits_dir / "X_test.csv")
y_test = pd.read_csv(splits_dir / "y_test.csv").squeeze("columns")

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")
print("Class balance in training set:")
print(y_train.value_counts(normalize=True))

## 2. Train and Evaluate All 5 ML Model Candidates

In [ ]:
candidates = get_model_candidates(random_state=42)
trained_models = {}
eval_results = []

for name, model in candidates.items():
    print(f"Training {name}...")
    fitted_model, duration = train_model(name, model, X_train, y_train, verbose=False)
    trained_models[name] = fitted_model
    metrics = evaluate_model(name, fitted_model, X_test, y_test, train_time_sec=duration)
    eval_results.append(metrics)
    print(f"  Done ({duration:.2f}s) - Accuracy: {metrics['accuracy']*100:.2f}%, F1: {metrics['f1_score']:.4f}")

comparison_df = pd.DataFrame(eval_results).sort_values("cybersecurity_score", ascending=False)
comparison_df[["model_name", "accuracy", "precision", "recall", "f1_score", "roc_auc", "training_time_seconds"]]

## 3. Visualize Model Comparisons & ROC Curves

In [ ]:
metrics_to_plot = ["accuracy", "precision", "recall", "f1_score", "roc_auc"]
plot_df = comparison_df.melt(id_vars=["model_name"], value_vars=metrics_to_plot, var_name="Metric", value_name="Score")

plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="Metric", y="Score", hue="model_name", palette="viridis")
plt.title("Comparison of Machine Learning Models Across Evaluation Metrics", fontsize=14, fontweight="bold")
plt.ylim(0.9, 1.0)
plt.grid(True, linestyle="--", alpha=0.5, axis="y")
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 4. Multi-Layer Risk Assessment Demonstration

In [ ]:
risk_engine = RiskAssessmentEngine(low_threshold=0.35, high_threshold=0.70)
best_model = trained_models[comparison_df.iloc[0]["model_name"]]

sample_test_urls = [
    "https://www.google.com/search?q=cybersecurity+defense",
    "http://192.168.1.100:8080/admin/secure_login.php",
    "http://paypal.com@attacker-stealer.net/verify/account",
    "https://bit.ly/3xYz123",
    "https://github.com/torvalds/linux"
]

for u in sample_test_urls:
    feats = extract_url_features(u)
    model_pred = predict_single_url_features(best_model, feats, FEATURE_COLUMNS)
    assessment = risk_engine.assess_risk(u, feats, model_pred)
    
    print(f"URL:            {u}")
    print(f"Classification: {assessment['classification']} (Probability: {assessment['phishing_probability']*100:.1f}%)")
    print(f"Risk Level:     {assessment['risk_level']} (Score: {assessment['risk_score']}/100)")
    print(f"Warnings ({len(assessment['warnings'])}):  {assessment['warnings'][:2]}")
    print("-" * 80)